# 2 — Build a clean, joined analysis table

**Snapshot:** the preparation code suggested after Notebook 1 has been reviewed and adapted.

This notebook is independently runnable. It validates the join, reconciles `MLL`/`LEP`, creates session-only student aliases, and reshapes the three seasons into a tidy table. It makes no network calls and writes no files.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)


def locate_data_file(filename):
    """Find a data file when the notebook starts in /notebooks or the project root."""
    for folder in (Path.cwd(), Path.cwd().parent):
        candidate = folder / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename!r} in the current folder or its parent.")


reading_path = locate_data_file("FakeiReadyData.csv")
math_path = locate_data_file("FakeiReadyMathData.csv")

# Read identifiers as strings so leading zeros would be preserved in a real export.
reading_raw = pd.read_csv(
    reading_path,
    dtype={"Student Id": "string", "Class": "string"},
)
math_raw = pd.read_csv(
    math_path,
    dtype={"Student Id": "string", "Class": "string"},
)


SEASONS = ["Fall", "Winter", "Spring"]


def assessment_column(subject, measure, season):
    """Build the exact multiline heading used by the source CSV files."""
    return f"I-Ready Result\n{subject}\n{measure} - {season}\n2025-2026"


# Validate the proposed one-to-one join without displaying identifiers.
reading_ids = set(reading_raw["Student Id"].dropna())
math_ids = set(math_raw["Student Id"].dropna())
id_sets_match = reading_ids == math_ids

reading_demo = reading_raw.set_index("Student Id")
math_demo = math_raw.set_index("Student Id").rename(columns={"LEP": "MLL"})
demo_fields = [
    "Student Name", "Gender", "SWD", "MLL", "ED", "Current Grade", "Class"
]
if id_sets_match:
    mismatch_count = int(
        (
            reading_demo[demo_fields].sort_index().astype("string")
            != math_demo[demo_fields].sort_index().astype("string")
        ).sum().sum()
    )
else:
    mismatch_count = -1

validation = pd.DataFrame(
    {
        "check": [
            "Reading IDs are present and unique",
            "Math IDs are present and unique",
            "Reading and math ID sets match",
            "Shared demographic fields agree",
        ],
        "passed": [
            reading_raw["Student Id"].notna().all()
            and not reading_raw["Student Id"].duplicated().any(),
            math_raw["Student Id"].notna().all()
            and not math_raw["Student Id"].duplicated().any(),
            id_sets_match,
            mismatch_count == 0,
        ],
    }
)
assert validation["passed"].all(), "Stop and resolve join validation failures before analysis."

# Build stable aliases for this session. This is pseudonymization, not anonymization.
key_map = {
    student_id: f"S{number:03d}"
    for number, student_id in enumerate(sorted(reading_ids), start=1)
}

demographics = (
    reading_raw.assign(student_key=reading_raw["Student Id"].map(key_map))
    .rename(
        columns={
            "Gender": "gender",
            "SWD": "swd",
            "MLL": "mll",
            "ED": "economically_disadvantaged",
            "Current Grade": "current_grade",
            "Class": "class_section",
        }
    )[
        [
            "student_key",
            "gender",
            "swd",
            "mll",
            "economically_disadvantaged",
            "current_grade",
            "class_section",
        ]
    ]
)


def extract_scores(raw, subject):
    """Return one row per student and season for one subject."""
    frames = []
    prefix = subject.lower()
    for season in SEASONS:
        selected = raw[
            [
                "Student Id",
                assessment_column(subject, "Overall SS", season),
                assessment_column(subject, "Percentile", season),
                assessment_column(subject, "Overall Placement", season),
            ]
        ].copy()
        selected.columns = [
            "student_id",
            f"{prefix}_score",
            f"{prefix}_percentile",
            f"{prefix}_placement",
        ]
        selected["student_key"] = selected["student_id"].map(key_map)
        selected["season"] = season
        frames.append(selected.drop(columns="student_id"))
    return pd.concat(frames, ignore_index=True)


reading_scores = extract_scores(reading_raw, "Reading")
math_scores = extract_scores(math_raw, "Math")

analysis = (
    reading_scores.merge(
        math_scores,
        on=["student_key", "season"],
        how="inner",
        validate="one_to_one",
    )
    .merge(demographics, on="student_key", how="left", validate="many_to_one")
)
analysis["season"] = pd.Categorical(
    analysis["season"], categories=SEASONS, ordered=True
)
analysis = analysis.sort_values(["student_key", "season"]).reset_index(drop=True)

quality_checks = pd.DataFrame(
    {
        "check": [
            "Expected student-season row count",
            "One row per student and season",
            "Every student has all three seasons",
            "No direct identifiers in analysis table",
            "No missing overall scores",
        ],
        "passed": [
            len(analysis) == len(reading_ids) * len(SEASONS),
            not analysis.duplicated(["student_key", "season"]).any(),
            analysis.groupby("student_key", observed=True)["season"].nunique().eq(3).all(),
            {"Student Id", "Student Name"}.isdisjoint(analysis.columns),
            analysis[["reading_score", "math_score"]].notna().all().all(),
        ],
    }
)
assert quality_checks["passed"].all(), "Stop and resolve preparation failures before analysis."

## Stop on join problems instead of silently creating bad results

In [2]:
validation

,check,passed
0,Reading IDs are present and unique,True
1,Math IDs are present and unique,True
2,Reading and math ID sets match,True
3,Shared demographic fields agree,True


## Inspect the analysis table—not the source identifiers

Each row is a student-season observation. `student_key` is useful inside this authorized local session, but it remains linkable and should still be treated as sensitive.

In [3]:
display(quality_checks)
display(analysis.head(9))

,check,passed
0,Expected student-season row count,True
1,One row per student and season,True
2,Every student has all three seasons,True
3,No direct identifiers in analysis table,True
4,No missing overall scores,True


,reading_score,reading_percentile,reading_placement,student_key,season,math_score,math_percentile,math_placement,gender,swd,mll,economically_disadvantaged,current_grade,class_section
0,315,2,Kindergarten,S001,Fall,305,1,Kindergarten,F,Yes,No,Yes,6,601
1,325,3,Kindergarten,S001,Winter,312,2,Kindergarten,F,Yes,No,Yes,6,601
2,339,4,Kindergarten,S001,Spring,324,3,Kindergarten,F,Yes,No,Yes,6,601
3,552,28,Grade 5,S002,Fall,550,28,Grade 5,M,No,No,Yes,6,602
4,556,29,Grade 5,S002,Winter,556,29,Grade 5,M,No,No,Yes,6,602
5,569,32,Grade 5,S002,Spring,568,32,Grade 5,M,No,No,Yes,6,602
6,602,42,Grade 6,S003,Fall,614,54,Grade 6,F,No,Yes,No,6,601
7,608,48,Grade 6,S003,Winter,621,61,Grade 6,F,No,Yes,No,6,601
8,617,57,Grade 6,S003,Spring,628,68,Grade 6,F,No,Yes,No,6,601


## A first aggregate summary

In [4]:
season_summary = (
    analysis.groupby("season", observed=True)
    .agg(
        students=("student_key", "nunique"),
        reading_mean=("reading_score", "mean"),
        math_mean=("math_score", "mean"),
        reading_median=("reading_score", "median"),
        math_median=("math_score", "median"),
    )
    .round(1)
    .reset_index()
)
season_summary

,season,students,reading_mean,math_mean,reading_median,math_median
0,Fall,60,530.0,527.5,571.0,566.0
1,Winter,60,536.9,534.0,577.5,571.0
2,Spring,60,544.1,542.7,583.0,579.5


## Why we do not save the tidy student-level table by default

A cleaned file with aliases and demographics is still sensitive and may be easier to misuse than the source export. Keep it in memory unless there is an approved storage need. If an export is required, use an approved encrypted location, access controls, retention limits, and a documented purpose.

## Prompt 2 — choose analyses that fit this longitudinal structure

Copy this text—not the `analysis` dataframe—to an approved assistant. This asks the model to reason from structure and safeguards rather than from student records.

In [5]:
prompt_2 = """
Help me plan exploratory analysis for a local pandas dataframe named `analysis`.
Do not ask for data, screenshots, outputs, or row samples. Student records must
stay local. There is one row per pseudonymous student and season.

Columns:
student_key; season (ordered Fall, Winter, Spring); reading_score;
reading_percentile; reading_placement; math_score; math_percentile;
math_placement; gender; swd; mll; economically_disadvantaged;
current_grade; class_section.

Please propose a short, presentation-friendly exploratory sequence and provide
separate Python code blocks for:
1. cohort mean Reading/Math score trends across seasons;
2. paired Fall-to-Spring growth distributions (not just cross-sectional means);
3. the relationship between baseline score and growth, with a warning about
   regression to the mean and no causal claims;
4. descriptive subgroup growth summaries for gender, SWD, MLL, and economic
   disadvantage, suppressing every result where n < 10.

Use pandas, matplotlib, and seaborn. Do not label student points, rank teachers
or classes, define 'success' without a supplied benchmark, or write files.
Include data-quality assertions and concise chart titles. Return rationale as
markdown, then code. The code will be reviewed and run only in Jupyter locally.
""".strip()

print(prompt_2)

Help me plan exploratory analysis for a local pandas dataframe named `analysis`.
Do not ask for data, screenshots, outputs, or row samples. Student records must
stay local. There is one row per pseudonymous student and season.

Columns:
student_key; season (ordered Fall, Winter, Spring); reading_score;
reading_percentile; reading_placement; math_score; math_percentile;
math_placement; gender; swd; mll; economically_disadvantaged;
current_grade; class_section.

Please propose a short, presentation-friendly exploratory sequence and provide
separate Python code blocks for:
1. cohort mean Reading/Math score trends across seasons;
2. paired Fall-to-Spring growth distributions (not just cross-sectional means);
3. the relationship between baseline score and growth, with a warning about
   regression to the mean and no causal claims;
4. descriptive subgroup growth summaries for gender, SWD, MLL, and economic
   disadvantage, suppressing every result where n < 10.

Use pandas, matplotlib, and s

### What to copy back

- Put the assistant's rationale into markdown cells.
- Put each `python` block into its own code cell so charts can be rerun independently.
- Reject any code that uploads data, installs unknown packages, writes unapproved files, or displays names/IDs.
- The next reviewed snapshot is **`03_exploratory_growth.ipynb`**.